# Week 4 — Final Evaluation

1,000-episode simulation harness for evaluating all pricing agents 
(heuristics, Q-Learning, DQN) across full booking seasons.

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
from pricing_env import PricingEnv
from baseline_agents import FixedPriceAgent, TimeBasedDiscountAgent, DemandBasedAgent


def run_large_scale_evaluation(agent, env, n_episodes=1000, has_reset=False):
    """
    Runs any agent (heuristic, Q-Learning, or DQN) across n_episodes 
    full booking seasons and returns detailed per-episode statistics.
    """
    episode_revenues = []
    episode_sell_through = []

    for ep in range(n_episodes):
        obs, info = env.reset()
        if has_reset:
            agent.reset()
        total_reward = 0
        initial_inventory = env.max_inventory

        done = False
        while not done:
            action = agent.act(obs)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            done = terminated or truncated

        episode_revenues.append(total_reward)
        remaining_inventory = obs[0]
        sell_through = (initial_inventory - remaining_inventory) / initial_inventory
        episode_sell_through.append(sell_through)

    return {
        "revenues": episode_revenues,
        "sell_through_rates": episode_sell_through,
        "mean_revenue": np.mean(episode_revenues),
        "std_revenue": np.std(episode_revenues),
        "mean_sell_through": np.mean(episode_sell_through)
    }

In [ ]:
env = PricingEnv()
test_agent = FixedPriceAgent()

results = run_large_scale_evaluation(test_agent, env, n_episodes=1000)
print(f"Mean Revenue: {results['mean_revenue']:.2f}")
print(f"Std Dev: {results['std_revenue']:.2f}")
print(f"Sell-Through Rate: {results['mean_sell_through']*100:.1f}%")

## DQN Multi-Season Price Trajectories

In [ ]:
import torch
from dqn_agent import DQNAgent
from plotting_utils import plot_multi_season_trajectories

dqn_agent = DQNAgent()
dqn_agent.policy_net.load_state_dict(torch.load('../outputs/dqn_checkpoints/dqn_ep2000.pt'))
dqn_agent.policy_net.eval()
dqn_agent.epsilon = 0.0  # greedy for evaluation

plot_multi_season_trajectories(
    dqn_agent, env, n_seasons=5, agent_type="dqn",
    title="Trained DQN — Price Trajectory Across 5 Sample Seasons",
    save_path='../outputs/dqn_multi_season_trajectories.png'
)

## Reward Sensitivity — Price Safety Bounds

In [ ]:
def evaluate_with_price_bounds(agent, env, min_bound, max_bound, n_episodes=100):
    episode_revenues = []
    for _ in range(n_episodes):
        obs, info = env.reset()
        total_reward = 0
        done = False
        while not done:
            raw_action = agent.act(obs)
            clipped_action = max(min_bound, min(raw_action, max_bound))
            obs, reward, terminated, truncated, info = env.step(clipped_action)
            total_reward += reward
            done = terminated or truncated
        episode_revenues.append(total_reward)
    return {"mean_revenue": np.mean(episode_revenues), "std_revenue": np.std(episode_revenues)}

bound_configs = [(0, 9), (2, 9), (2, 7), (4, 9)]

sensitivity_results = []
for min_b, max_b in bound_configs:
    result = evaluate_with_price_bounds(dqn_agent, env, min_b, max_b, n_episodes=100)
    sensitivity_results.append({"min_bound": min_b, "max_bound": max_b, **result})
    print(f"Bounds [{min_b},{max_b}] -> mean revenue: {result['mean_revenue']:.2f}")

## Run All Agents Across 1,000 Simulated Booking Seasons

In [ ]:
from random_agent import RandomAgent
from baseline_agents import FixedPriceAgent, TimeBasedDiscountAgent, DemandBasedAgent
from q_learning_agent import QLearningAgent
from dqn_agent import DQNAgent
import torch

q_agent_loaded = QLearningAgent()
q_agent_loaded.load('../outputs/trained_qtable_best.npy')
q_agent_loaded.epsilon = 0.0

dqn_agent = DQNAgent()
dqn_agent.policy_net.load_state_dict(torch.load('../outputs/dqn_checkpoints/dqn_ep2000.pt'))
dqn_agent.policy_net.eval()
dqn_agent.epsilon = 0.0

all_agents = {
    "Random": RandomAgent(env.action_space),
    "Fixed": FixedPriceAgent(),
    "Discount": TimeBasedDiscountAgent(),
    "Demand": DemandBasedAgent(),
    "Q-Learning": q_agent_loaded,
    "DQN": dqn_agent,
}

full_results = {}
for name, a in all_agents.items():
    full_results[name] = run_large_scale_evaluation(a, env, n_episodes=1000)
    print(f"{name}: mean_revenue={full_results[name]['mean_revenue']:.2f}, "
          f"sell_through={full_results[name]['mean_sell_through']*100:.1f}%")

## Revenue Distribution Comparison — Violin Plot

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
data = [full_results[name]["revenues"] for name in full_results]
labels = list(full_results.keys())

parts = plt.violinplot(data, showmeans=True)
plt.xticks(range(1, len(labels) + 1), labels)
plt.title("Revenue-Per-Episode Distribution — All Agents (1000 episodes each)")
plt.ylabel("Episodic Revenue")
plt.tight_layout()
plt.savefig('../outputs/all_agents_violin_comparison.png', dpi=150)
plt.show()

## Agent Robustness Under Varying Demand Elasticity

In [ ]:
class ElasticPricingEnv(PricingEnv):
    """Wraps PricingEnv with an adjustable price sensitivity multiplier."""
    def __init__(self, elasticity_multiplier=1.0, **kwargs):
        super().__init__(**kwargs)
        self.elasticity_multiplier = elasticity_multiplier

    def _compute_demand(self, price_level, days_until_departure):
        price_sensitivity = 5.0 * self.elasticity_multiplier
        urgency = 1 - (days_until_departure / self.max_days)
        z = -price_sensitivity * price_level + 2 * urgency
        purchase_probability = 1 / (1 + np.exp(-z))
        expected_units = purchase_probability * 10
        return np.random.poisson(lam=expected_units)


elasticity_levels = {"Low sensitivity": 0.5, "Base": 1.0, "High sensitivity": 2.0}
elasticity_results = {}

for elab, mult in elasticity_levels.items():
    test_env = ElasticPricingEnv(elasticity_multiplier=mult)
    elasticity_results[elab] = {}
    for name, a in all_agents.items():
        r = run_large_scale_evaluation(a, test_env, n_episodes=200)
        elasticity_results[elab][name] = r["mean_revenue"]

results_df = pd.DataFrame(elasticity_results)
results_df